# Oasis Infobyte Data Analytics Internship
## Project 3: Credit Card Fraud Detection using Machine Learning
**Author:** Jasmin Jamadar  
**Role:** Data Analytics Intern  
**Project Objective:** Build a comparative machine learning pipeline that addresses severe class imbalance to accurately identify fraudulent credit card transactions.

---



### Project Outline
1. **Environment Setup & Libraries** - Load data analysis, machine learning, and visualization libraries.
2. **Dataset Loading & Initial Ingestion** - Loading `creditcard.csv` and inspecting metadata.
3. **Data Cleaning & Quality Control** - Checking for null values and removing duplicates.
4. **Exploratory Data Analysis (EDA)** - Visualizing transaction amounts and extreme class imbalance.
5. **Preprocessing & Scaling** - Partitioning data via stratified train-test split and scaling `Time`/`Amount` features.
6. **Imbalance Treatment (Under-sampling)** - Creating a balanced training set to prevent model bias.
7. **Model Training & Comparison** - Building and training **Logistic Regression** and **Random Forest Classifiers**.
8. **Evaluation Metrics** - Computing Accuracy, Precision, Recall, F1, and ROC-AUC on the imbalanced test set.
9. **Diagnostics & Visualizations** - Generating confusion matrices, ROC curves, and feature importance.
10. **Analytical Insights & Business Prevention Guide** - Actionable fraud prevention recommendations.



### 1. Environment Setup & Library Imports
We begin by loading standard data science packages for preprocessing, modeling, and plotting.



In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, confusion_matrix, roc_curve, classification_report)

# Configure visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9
})
print("Environment setup complete.")


### 2. Dataset Loading & Initial Ingestion
We load the dataset `creditcard.csv` from the `Dataset` folder and inspect its dimensions.



In [ ]:
# Define dataset path
dataset_path = os.path.join("..", "Dataset", "creditcard.csv")

# Load data
df = pd.read_csv(dataset_path)
print(f"Dataset successfully loaded. Shape: {df.shape}")
print("\n--- Columns in Dataset ---")
print(df.columns.tolist()[:10] + ['...'] + df.columns.tolist()[-5:])
print("\n--- Dataset Head ---")
print(df.head())


### 3. Data Cleaning & Quality Control
We check for missing values, verify data types, and remove duplicate transactions.



In [ ]:
# Missing value check
print("Total missing values in dataset:", df.isnull().sum().sum())

# Duplicates check and removal
dupes = df.duplicated().sum()
print(f"Duplicate rows found: {dupes}")
if dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicates removed. New dataset shape: {df.shape}")


### 4. Exploratory Data Analysis (EDA)
We visualize the extreme class imbalance and analyze the distribution of transaction amounts for legitimate vs. fraudulent transactions.



In [ ]:
# Check class counts
counts = df['Class'].value_counts()
print("Class Counts:")
print(counts)
print(f"Fraud Rate: {counts[1]/len(df)*100:.4f}%")

# Plot Class distribution
plt.figure(figsize=(7, 5))
ax = sns.barplot(x=counts.index, y=counts.values, palette='coolwarm', hue=counts.index, legend=False)
plt.title('Transaction Class Distribution (Log Scale)')
plt.xlabel('Class (0: Legitimate, 1: Fraud)')
plt.ylabel('Count (Log Scale)')
plt.yscale('log')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontweight='bold', xytext=(0, 5), textcoords='offset points')
os.makedirs(os.path.join("..", "Visualizations"), exist_ok=True)
plt.savefig(os.path.join("..", "Visualizations", "fraud_distribution.png"), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Plot Transaction Amount Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Legitimate
sns.histplot(df[df['Class'] == 0]['Amount'] + 1, log_scale=True, kde=True, color='#3b82f6', ax=axes[0])
axes[0].set_title('Legitimate Transaction Amount Distribution')
axes[0].set_xlabel('Amount + 1 (log10)')

# Fraudulent
sns.histplot(df[df['Class'] == 1]['Amount'] + 1, log_scale=True, kde=True, color='#ef4444', ax=axes[1])
axes[1].set_title('Fraudulent Transaction Amount Distribution')
axes[1].set_xlabel('Amount + 1 (log10)')

plt.tight_layout()
plt.savefig(os.path.join("..", "Visualizations", "transaction_amount_distribution.png"), dpi=150, bbox_inches='tight')
plt.show()


### 5. Preprocessing & Feature Scaling
We partition the dataset into training (80%) and testing (20%) sets using stratified split to preserve class ratios.
Since `Time` and `Amount` have large variances and outliers, we scale them using `RobustScaler`.



In [ ]:
# Features and target split
X = df.drop(columns=['Class'])
y = df['Class']

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Training set: {X_train.shape[0]} rows, Test set: {X_test.shape[0]} rows")

# Scaling continuous variables
scaler = RobustScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scale_cols = ['Time', 'Amount']
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])
print("Scaling complete.")


### 6. Imbalance Treatment: Random Under-sampling
To prevent the classifiers from being heavily biased toward predicting the majority class (legitimate), we sub-sample legitimate records in the training set to match the count of fraud records, creating a 50/50 balanced training split.



In [ ]:
# Extract fraud indices in training
fraud_indices = y_train[y_train == 1].index
num_frauds = len(fraud_indices)

# Extract non-fraud indices in training
non_fraud_indices = y_train[y_train == 0].index

# Randomly sample non-frauds
np.random.seed(42)
sample_non_fraud_indices = np.random.choice(non_fraud_indices, num_frauds, replace=False)

# Combine indices
balanced_indices = np.concatenate([fraud_indices, sample_non_fraud_indices])

# Select training data
X_train_balanced = X_train_scaled.loc[balanced_indices]
y_train_balanced = y_train.loc[balanced_indices]

# Shuffle training split
shuffle_idx = np.random.permutation(len(y_train_balanced))
X_train_balanced = X_train_balanced.iloc[shuffle_idx]
y_train_balanced = y_train_balanced.iloc[shuffle_idx]

print(f"Balanced training split shape: {X_train_balanced.shape}")
print(y_train_balanced.value_counts())


### 7. Model Training & Comparison
We train **Logistic Regression** and **Random Forest Classifiers** on the balanced training data, and predict on the imbalanced test set.



In [ ]:
results = {}
confusion_matrices = {}
roc_curves = {}

# --- 1. Logistic Regression ---
print("Training Logistic Regression...")
t_start = time.time()
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_balanced, y_train_balanced)
t_lr = time.time() - t_start

lr_pred = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

# Store results
results['Logistic Regression'] = {
    'Accuracy': accuracy_score(y_test, lr_pred),
    'Precision': precision_score(y_test, lr_pred),
    'Recall': recall_score(y_test, lr_pred),
    'F1-Score': f1_score(y_test, lr_pred),
    'ROC-AUC': roc_auc_score(y_test, lr_proba)
}
confusion_matrices['Logistic Regression'] = confusion_matrix(y_test, lr_pred)
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
roc_curves['Logistic Regression'] = (fpr_lr, tpr_lr)

print("Logistic Regression evaluation complete.")


In [ ]:
# --- 2. Random Forest Classifier ---
print("Training Random Forest Classifier...")
t_start = time.time()
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train_balanced, y_train_balanced)
t_rf = time.time() - t_start

rf_pred = rf_model.predict(X_test_scaled)
rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

# Store results
results['Random Forest Classifier'] = {
    'Accuracy': accuracy_score(y_test, rf_pred),
    'Precision': precision_score(y_test, rf_pred),
    'Recall': recall_score(y_test, rf_pred),
    'F1-Score': f1_score(y_test, rf_pred),
    'ROC-AUC': roc_auc_score(y_test, rf_proba)
}
confusion_matrices['Random Forest Classifier'] = confusion_matrix(y_test, rf_pred)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
roc_curves['Random Forest Classifier'] = (fpr_rf, tpr_rf)

print("Random Forest evaluation complete.")


In [ ]:
# Tabulate and print results
metrics_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Model'})
print(metrics_df)


In [ ]:
# Plot side-by-side Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
classes = ['Legitimate (0)', 'Fraudulent (1)']

for idx, (model_name, cm) in enumerate(confusion_matrices.items()):
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=axes[idx], cbar=False,
                xticklabels=classes, yticklabels=classes,
                annot_kws={"size": 13, "weight": "bold"})
    
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[idx].text(j + 0.5, i + 0.7, f"({cm_norm[i, j]*100:.2f}%)",
                           ha='center', va='center', color='darkslategray', fontsize=10)
            
    axes[idx].set_title(f'{model_name} Confusion Matrix', fontsize=14, fontweight='bold', pad=10)
    axes[idx].set_xlabel('Predicted Class')
    axes[idx].set_ylabel('Actual Class')

plt.tight_layout()
plt.savefig(os.path.join("..", "Visualizations", "confusion_matrix.png"), dpi=150, bbox_inches='tight')
plt.show()

# Plot ROC Curves
plt.figure(figsize=(8, 6))
for model_name, (fpr, tpr) in roc_curves.items():
    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {results[model_name]['ROC-AUC']:.4f})", linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.5000)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve Comparison')
plt.legend(loc="lower right")
plt.savefig(os.path.join("..", "Visualizations", "roc_curve.png"), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Feature correlation matrix of entire dataset (to save as chart)
plt.figure(figsize=(15, 12))
sns.heatmap(df.corr(), cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.1)
plt.title('Complete Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.savefig(os.path.join("..", "Visualizations", "correlation_heatmap.png"), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Extract Random Forest Feature Importances
importances = rf_model.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot top 12 features
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_imp.head(12), palette='mako', hue='Feature', legend=False)
plt.title('Top 12 Most Important Features (Random Forest)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.savefig(os.path.join("..", "Visualizations", "feature_importance.png"), dpi=150, bbox_inches='tight')
plt.show()


### 8. Key Findings & Business Recommendations

#### Analytical Summary
1. **Recall Priority**: For fraud detection, **Recall** is the crucial metric, as failing to identify a fraudulent transaction leads to direct financial loss. **Random Forest** achieved a Recall of **86.32%** on the imbalanced test set, while **Logistic Regression** achieved **87.37%**.
2. **False Alarms (Precision)**: Random Forest significantly outperformed Logistic Regression in minimizing false alarms. Random Forest's Precision (**10.96%**) was double that of Logistic Regression (**5.08%**), cutting the volume of false alerts in half.
3. **Key Indicators**: Gini feature importances indicate that `V14`, `V10`, `V17`, and `V12` are the most influential latent variables driving fraud detection.

#### Business Recommendations
* **Tiered Alerts**: Implement a tiered risk alert system. Transactions flagged with high probability by Random Forest should block the card immediately, while lower probability matches prompt a quick two-factor SMS validation.
* **Feature Focus**: Prioritize real-time monitoring of transactions exhibiting abnormalities in V14, V10, and V12 latent variables.
* **Balance Cost**: Evaluate the operational cost of managing false alarms (666 false alarms in Random Forest) vs. the cost of a missed fraud event (~13 missed frauds).

---
*End of Notebook. Submission-ready for Oasis Infobyte Data Analytics Internship.*

